In [1]:
import evaluate
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

In [2]:
# Verify GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on device: {device}")
if device == "cuda":
  print(f"GPU: {torch.cuda.get_device_name(0)}")

Training on device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


In [3]:
# Load PubMedQA dataset
raw_dataset = load_dataset("pubmed_qa", "pqa_labeled", split="train")

label_map = {"yes": 0, "no": 1, "maybe": 2}
id2label = {0: "yes", 1: "no", 2: "maybe"}
label2id = {v: k for k, v in id2label.items()}


def format_dataset(batch):
  formatted_texts = [
      f"Question: {q} Context: {' '.join(c['contexts'])}"
      for q, c in zip(batch["question"], batch["context"])
  ]
  labels = [label_map[ans] for ans in batch["final_decision"]]
  return {"text": formatted_texts, "label": labels}


processed_data = raw_dataset.map(format_dataset, batched=True)
split_dataset = processed_data.train_test_split(test_size=0.2, seed=42)
print("Dataset loaded and split successfully!")

Dataset loaded and split successfully!


In [4]:
MODEL_NAME = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_function(examples):
  return tokenizer(
      examples["text"], padding="max_length", truncation=True, max_length=256
  )


tokenized_train = split_dataset["train"].map(tokenize_function, batched=True)
tokenized_eval = split_dataset["test"].map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
).to(device)

print("SciBERT weights and tokenizer initialized!")

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SciBERT weights and tokenizer initialized!


In [5]:
metric_f1 = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")


def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  acc = metric_acc.compute(predictions=predictions, references=labels)[
      "accuracy"
  ]
  f1 = metric_f1.compute(
      predictions=predictions, references=labels, average="macro"
  )["f1"]
  return {"accuracy": acc, "macro_f1": f1}


training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
    logging_steps=20,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.958900,0.954108,0.545000,0.305795
2,0.880400,0.923712,0.575000,0.403622
3,0.763100,0.916625,0.615000,0.420121


TrainOutput(global_step=150, training_loss=0.8514879353841146, metrics={'train_runtime': 45.163, 'train_samples_per_second': 53.141, 'train_steps_per_second': 3.321, 'total_flos': 315736101273600.0, 'train_loss': 0.8514879353841146, 'epoch': 3.0})

In [ ]:
model.save_pretrained("./saved_model")
tokenizer.save_pretrained("./saved_model")
print("Model and Tokenizer successfully saved to ./saved_model!")

Model and Tokenizer successfully saved to ./saved_model!
